- set of probes constant
- code of call always 6, 7, 8, 11
- 別に特別な模様は観測されない

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans Mono'
import json

In [30]:
import os

calls_raw = []
labels = []

data_path = 'E:/Gene Expression Omnibus/extracted/GSE33356' 
for file_name in os.listdir(data_path):
    file_r = open(os.path.join(data_path, file_name), 'r')
    json_file = json.load(file_r)
    file_r.close()
    
    # assert json_file['MultiDataTypeCounts']['Genotype'] == 909622
    # assert json_file['Genotype']['ProbeNames'] == probe_names, file_name
    # assert set(json_file['Genotype']['Call']).issubset({6, 7, 8, 11})

    calls_raw.append( json_file['Genotype']['Call'] )
    labels.append( 'N' if 'N' in file_name else 'T')
probe_names = json_file['Genotype']['ProbeNames']
del json_file

calls_raw = np.array(calls_raw, dtype=np.float32)
labels = np.array(labels)

In [99]:
from sklearn.impute import SimpleImputer

calls = calls_raw
mask_too_much_missing = (np.mean(calls_raw == 11, axis=0) > 0.1)
calls = calls[:, ~mask_too_much_missing]
calls = SimpleImputer(missing_values=11., strategy='most_frequent').fit_transform(calls)
mask_few_minor_allele = (np.mean((calls==8)+0.5*(calls==7), axis=0) < 0.1) | (np.mean((calls==6)+0.5*(calls==7), axis=0) < 0.1)
mask_hetero_only = np.all(calls == 7, axis=0)
calls = calls[:, ~(mask_few_minor_allele | mask_hetero_only)]

c:\Users\sinwa\AppData\Local\Programs\Python\Python38\lib\site-packages\sklearn\impute\_base.py:49: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode = stats.mode(array)


In [276]:
def orbital_distance(genotype_array, batch_size=256):
    result = np.zeros((genotype_array.shape[1], )*2)
    for i in range(0, genotype_array.shape[1], batch_size):
        dist_same_order = np.mean(genotype_array[:, i:i+batch_size, np.newaxis] != genotype_array[:, np.newaxis, :], axis=0)
        dist_rev_order = np.mean(genotype_array[:, i:i+batch_size, np.newaxis] != (14-genotype_array[:, np.newaxis, :]), axis=0)
        result[i:i+batch_size] = np.minimum(dist_same_order, dist_rev_order)
    return result

In [100]:
rng = np.random.default_rng(42)
mask_small = np.full((calls.shape[1], ), False)
mask_small[rng.choice(calls.shape[1], 10000, replace=False)] = True

calls_small = calls[:, mask_small]
dist_small = orbital_distance(calls_small)
# dist_small = np.zeros((calls_small.shape[1], )*2)
# for i in range(calls_small.shape[1]):
#     dist_same_order = np.mean(calls_small[:, i:i+1, np.newaxis] != calls_small[..., np.newaxis, :], axis=0)
#     dist_rev_order = np.mean(calls_small[:, i:i+1, np.newaxis] != (14-calls_small[..., np.newaxis, :]), axis=0)
#     dist_small[i] = np.minimum(dist_same_order, dist_rev_order)
#     if (i+1)%1000 == 0:
#         print(i+1, end=' ')

1000 2000 3000 4000 5000 6000 7000 8000 9000 10000 

In [101]:
entropies_small = np.zeros((calls_small.shape[1], ))
for val in [6, 7, 8]:
    prob = np.mean(calls_small == val, axis=0)
    entropies_small -= prob*np.log(prob, where=prob>0)

In [113]:
from gtda.homology import VietorisRipsPersistence

pers_hom = VietorisRipsPersistence(metric='precomputed', homology_dimensions=(0, 1,))
Xt = pers_hom.fit_transform(dist_small[np.newaxis])

In [158]:
from gtda.diagrams import PairwiseDistance

rng = np.random.default_rng(42)
random_idxs = rng.choice(dist_small.shape[0], 100, replace=False)
Xt_c = pers_hom.fit_transform(dist_small[np.newaxis, random_idxs][..., random_idxs])
pdist = PairwiseDistance(metric='silhouette')

GROUP LASSO????

In [283]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score

from gtda.diagrams import PairwiseDistance
from gtda.homology import VietorisRipsPersistence

def get_model():
    model = make_pipeline(
        OrdinalEncoder(
            categories=[[6., 7., 8.]] * X.shape[1]
        ),
        CategoricalNB(
            min_categories=3,
        )
    )
    return model

# splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
splitter = RepeatedStratifiedKFold(n_splits=5, n_repeats=500, random_state=42)

y = labels
# rng = np.random.default_rng(42)
# y = np.full(labels.shape, False)
# y[rng.choice(np.where(labels=='T')[0], 30, replace=False)] = True
# y[rng.choice(np.where(labels=='N')[0], 31, replace=False)] = True
pdists = [
    PairwiseDistance(metric='silhouette').fit(Xt),
    PairwiseDistance(metric='landscape').fit(Xt),
    PairwiseDistance(metric='bottleneck').fit(Xt)
]
pers_hom = VietorisRipsPersistence(metric='precomputed', homology_dimensions=(0, 1,))

scores_after = []; scores_before = []
dists_top = []; dists_conv = []
rng = np.random.default_rng(42)

for train_idxs, test_idxs in splitter.split(calls_small, y):
# for no in range(50000):
    random_idxs = rng.choice(calls_small.shape[1], 100, replace=False)

    snp_dists_train_only = orbital_distance(calls_small[train_idxs])
    snp_barcodes = pers_hom.fit_transform(snp_dists_train_only[np.newaxis])
    snp_barcodes_sub = pers_hom.fit_transform(snp_dists_train_only[np.newaxis, random_idxs][..., random_idxs])

    dists_top.append([
        pdist.fit(snp_barcodes).transform(snp_barcodes_sub)[0, 0]
        for pdist in pdists
    ])
    dists_conv.append([
        np.max(np.min(np.delete(snp_dists_train_only[random_idxs], random_idxs, axis=1), axis=0))
    ])

    for X, scores in zip([calls_small, calls_small[:, random_idxs]], [scores_before, scores_after]):
        model = get_model()
        model.fit(X[train_idxs], y[train_idxs])
        scores.append([
            1e2*accuracy_score(y[test_idxs], model.predict(X[test_idxs])),
            1e2*roc_auc_score(y[test_idxs], model.predict_proba(X[test_idxs])[..., 1])
        ])
    # break
scores_after = np.array(scores_after); scores_before = np.array(scores_before)
dists_top = np.array(dists_top); dists_conv = np.array(dists_conv)

In [287]:
dists_top.shape, dists_conv.shape, scores_after.shape, scores_before.shape

((2500, 3), (2500, 1), (2500, 2), (2500, 2))

In [302]:
from scipy.stats import pearsonr, spearmanr

for corr in [pearsonr, spearmanr]:
    print(
        corr(
            (scores_after-scores_before)[:, 1], 
            dists_top[:, 0]
            # dists_conv[:, 0]
        )
    )

# plt.scatter(scores[:, 1], dists, alpha=0.1)

PearsonRResult(statistic=-0.05987635595268212, pvalue=0.0027443777385273726)
SpearmanrResult(correlation=-0.04724517859103116, pvalue=0.01815720033371214)
